# Workspace Ownership Lookup

Maps GCP project IDs to the Workbench workspace behind them, showing who created and owns each workspace.

## Configuration

In [ ]:
# ---- CONFIGURATION ----

# Your org's user-facing ID (used as suffix on table names)
ORG_SUFFIX = "CHANGE_ME"

# Environment suffix
ENV = "prod" 

# GCP project where the log sink tables live
LOG_SINK_PROJECT = "workbench-bq-log-sink"

# ---- GCP PROJECT IDS TO LOOK UP ----
# Paste one or more GCP project IDs from the Billing Console
GCP_PROJECT_IDS = [
    "wb-example-project-1234",
    # "wb-another-project-5678",
]

In [ ]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client()
org_logs_dataset = f"{LOG_SINK_PROJECT}.workbench_monitoring_org_logs_{ENV}"
project_list = ", ".join([f"'{p}'" for p in GCP_PROJECT_IDS])
print(f"Looking up {len(GCP_PROJECT_IDS)} GCP project(s)...")

## Workspace Details

Displays the workspace details behind each GCP project, who created it, and when.

In [ ]:
ws_query = f"""
SELECT
    gcp_project_id,
    workspace_id,
    workspace_user_facing_id,
    workspace_display_name,
    created_date,
    created_by_email,
    state,
    is_data_collection
FROM `{org_logs_dataset}.wsm_workspaces_{ORG_SUFFIX}`
WHERE gcp_project_id IN ({project_list})
ORDER BY gcp_project_id
"""

ws_df = client.query(ws_query).to_dataframe()

if ws_df.empty:
    print("No workspaces found for the provided GCP project ID(s).")
else:
    print(f"Found {len(ws_df)} workspace(s)")
    display(ws_df)

## Workspace Owners

One row per owner — shows every individual and group with the OWNER role on each workspace (consists of those with individual ownership grant and access via owner group)

In [ ]:
if ws_df.empty:
    print("No workspaces found — skipping owner lookup.")
else:
    ws_ids = ", ".join([f"'{w}'" for w in ws_df['workspace_id'].tolist()])

    owners_query = f"""
    SELECT
        w.gcp_project_id,
        w.workspace_user_facing_id,
        w.workspace_display_name,
        p.user_email AS owner_email,
        p.grant_type
    FROM `{org_logs_dataset}.workspace_policy_grants_{ORG_SUFFIX}` p
    INNER JOIN `{org_logs_dataset}.wsm_workspaces_{ORG_SUFFIX}` w
        ON p.workspace_id = w.workspace_id
    WHERE p.role = 'OWNER'
        AND p.workspace_id IN ({ws_ids})
    ORDER BY w.gcp_project_id, p.user_email
    """

    owners_df = client.query(owners_query).to_dataframe()

    if owners_df.empty:
        print("No owners found for these workspaces.")
    else:
        print(f"{len(owners_df)} owner grant(s) across {owners_df['workspace_user_facing_id'].nunique()} workspace(s)")
        display(owners_df)

## Export to CSV

In [ ]:
if not ws_df.empty:
    ws_df.to_csv('workspace_details.csv', index=False)
    print('Saved to workspace_details.csv')

if 'owners_df' in dir() and not owners_df.empty:
    owners_df.to_csv('workspace_owners.csv', index=False)
    print('Saved to workspace_owners.csv')